# Detecção de Idioma dos Vídeos

Este notebook filtra vídeos em português usando detecção automática de idioma baseada em modelo transformer.

**Pipeline**: Concatenação (título + descrição) → Detecção de idioma (XLM-RoBERTa) → Filtragem de vídeos em português

**Resultado**: Lista de IDs de vídeos classificados como português para análise posterior

In [1]:
import pandas as pd
import numpy as np
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer
import torch
from tqdm import tqdm

In [2]:
df = pd.read_csv('cleaned_data/filtered_videos.csv')
df.head()

,video_id,title,description,channel_id,published_at,category_id,tags,view_count,like_count,comment_count,...,scheduled_end_time,concurrent_viewers,active_live_chat_id,recording_date,topicCategories,processing_status,parts_total,parts_processed,time_left_ms,processing_failure_reason
0,g3YDGLa2pWg,1 DURATESTON por semana: Dá para FICAR GRANDE?,1 DURATESTON por semana: Dá para FICAR GRANDE?...,UCYgV9EbxVSNdDHBr0GWLCmQ,2024-12-31,27,"['1 durateston por semana', '1 dura por semana...",6890,684,99,...,NaN,0,NaN,NaN,"['https://en.wikipedia.org/wiki/Health', 'http...",NaN,0,0,0,NaN
1,BMIQPwdZvN0,1ml DURATESTON por semana e REPOSIÇÃO ou CICLO...,De fã para fã,UCbzPEdQRN97S-VvDGXWwh4Q,2024-12-27,22,"['academia', 'musculação', 'fitness']",2436,164,11,...,NaN,0,NaN,NaN,['https://en.wikipedia.org/wiki/Lifestyle_(soc...,NaN,0,0,0,NaN
2,yvzlKwX8IFI,FERRITINA BAIXA COM DURATESTON: ISSO É NORMAL?,"Neste vídeo, respondo a uma dúvida intrigante ...",UCu18IKKSh83CIYzKilUCtUA,2024-12-08,27,"['Jorge Yamamoto', 'Dr Jorge Yamamoto', 'Testo...",1366,169,7,...,NaN,0,NaN,NaN,['https://en.wikipedia.org/wiki/Health'],NaN,0,0,0,NaN
3,86DW4tTw2BQ,REPRISE - Aula #161 - Desvendando os mecanism...,Me siga também no Instagram: https://www.insta...,UChT315XMRj2MZ7xMOYhfWtQ,2024-12-31,27,[],3812,406,6,...,NaN,0,NaN,NaN,"['https://en.wikipedia.org/wiki/Health', 'http...",NaN,0,0,0,NaN
4,grb8otb7UpA,A melhor forma de dividir a dose de testostero...,A melhor forma de dividir a dose de testostero...,UCYgV9EbxVSNdDHBr0GWLCmQ,2024-12-03,27,"['Melhor forma de fracionar a testosterona', '...",4183,464,72,...,NaN,0,NaN,NaN,"['https://en.wikipedia.org/wiki/Health', 'http...",NaN,0,0,0,NaN


In [3]:
df.shape

(11799, 35)

In [4]:
def text_concat(row):
    titulo = str(row['title']) if pd.notna(row['title']) else ''
    descricao = str(row['description']) if pd.notna(row['description']) else ''
    
    full_text = f"{titulo} {descricao}"
    
    return full_text

---
## 1. Preparação dos Dados

Carrega o dataset de vídeos e prepara o texto para análise:
- **Concatenação**: Combina título e descrição em um único campo de texto
- **Normalização**: Converte texto para lowercase
- **Tokenização**: Calcula número de tokens (máximo 512 para o modelo)

In [5]:
model_ckpt = 'papluca/xlm-roberta-base-language-detection'
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModelForSequenceClassification.from_pretrained(model_ckpt)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

XLMRobertaForSequenceClassification(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=

In [6]:
new_df = df[['video_id', 'title', 'description']].copy()
new_df['full_text'] = new_df.apply(text_concat, axis=1).str.lower()
new_df.shape

(11799, 4)

In [7]:
new_df['num_tokens'] = new_df['full_text'].apply(lambda x: len(tokenizer.encode(x, truncation=True, max_length=512)))
new_df['num_tokens'].describe()

count    11799.000000
mean       175.150691
std        169.078024
min          4.000000
25%         30.000000
50%        117.000000
75%        276.000000
max        512.000000
Name: num_tokens, dtype: float64

In [8]:
new_df.head()

,video_id,title,description,full_text,num_tokens
0,g3YDGLa2pWg,1 DURATESTON por semana: Dá para FICAR GRANDE?,1 DURATESTON por semana: Dá para FICAR GRANDE?...,1 durateston por semana: dá para ficar grande?...,449
1,BMIQPwdZvN0,1ml DURATESTON por semana e REPOSIÇÃO ou CICLO...,De fã para fã,1ml durateston por semana e reposição ou ciclo...,33
2,yvzlKwX8IFI,FERRITINA BAIXA COM DURATESTON: ISSO É NORMAL?,"Neste vídeo, respondo a uma dúvida intrigante ...",ferritina baixa com durateston: isso é normal?...,512
3,86DW4tTw2BQ,REPRISE - Aula #161 - Desvendando os mecanism...,Me siga também no Instagram: https://www.insta...,reprise - aula #161 - desvendando os mecanism...,48
4,grb8otb7UpA,A melhor forma de dividir a dose de testostero...,A melhor forma de dividir a dose de testostero...,a melhor forma de dividir a dose de testostero...,468


In [9]:
BATCH_SIZE = 16

file_to_save = 'cleaned_data/ids_portuguese.txt'

portuguese_video_ids = []

all_full_texts = new_df['full_text'].tolist()
all_video_ids = new_df['video_id'].tolist()

---
## 2. Detecção de Idioma

Classifica o idioma de cada vídeo usando **XLM-RoBERTa** (modelo multilíngue):
- **Modelo**: papluca/xlm-roberta-base-language-detection (suporta 20+ idiomas)
- **Processamento**: Batch size de 16 para otimizar uso de GPU
- **Pipeline**: Tokenização → Inferência → Softmax → Classificação
- **Filtro**: Seleciona apenas vídeos classificados como 'pt' (português)
- **Output**: Arquivo de texto com IDs de vídeos em português (ids_portuguese.txt)

In [10]:
for i in tqdm(range(0, len(all_full_texts), BATCH_SIZE), desc="Detectando idiomas"):
    batch_texts = all_full_texts[i : i + BATCH_SIZE]
    batch_video_ids = all_video_ids[i : i + BATCH_SIZE]

    inputs = tokenizer(batch_texts, truncation=True, padding=True, return_tensors="pt").to(device)

    with torch.no_grad():
        logits = model(**inputs).logits

    preds = torch.softmax(logits, dim=-1)

    id2lang = model.config.id2label
    predicted_lang_ids = torch.argmax(preds, dim=1).cpu().numpy()

    for j, predicted_lang_id in enumerate(predicted_lang_ids):
        video_id = batch_video_ids[j]
        predicted_lang = id2lang[predicted_lang_id]
        
        if predicted_lang == 'pt':
            portuguese_video_ids.append(str(video_id))

Detectando idiomas: 100%|██████████| 738/738 [08:34<00:00,  1.44it/s]


In [11]:
with open(file_to_save, 'w') as f:
    for video_id in portuguese_video_ids:
        f.write(f"{video_id}\n")

In [12]:
print('Total de vídeos no dataset:', len(df))
print('Total de vídeos em português no dataset:', len(portuguese_video_ids))
print('Proporção de vídeos em português no dataset:', len(portuguese_video_ids) / len(df))
print('Arquivo salvo:', file_to_save)

Total de vídeos no dataset: 11799
Total de vídeos em português no dataset: 6974
Proporção de vídeos em português no dataset: 0.5910670395796254
Arquivo salvo: cleaned_data/ids_portuguese.txt
